# 04 · Answer key, land-map arm, D0 and "places like this point"

For each question and pool (its own area's test tiles, and the held-out state):
- **truth** = the hand-written reference plan on the **official maps** plus NASA POWER (comparison arm (b));
- **land-map arm** = the same reference plan on the **AlphaEarth maps** from notebook 03 (isolates land-map errors);
- **D0** = the same reference plan on maps from **planted** vectors;
- **"places like this point"** (information) = AlphaEarth and POWER-climate similarity to 5 answer pixels taken
  from the training tiles of the question's own area.

F1 over all valid pixels of the pool's test tiles. Pass marks in `PLAN.md` (amendment A2).

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.score import f1
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
from sklearn.metrics import average_precision_score

from askmap.models import like_point_scores

LT = np.load(ROOT / "climate" / "long_term_z.npy")
LP = cfg["like_point"]
AE = {}
for area in POOL_Q:
    for n in TILES[area]["train"] + TILES[area]["test"]:
        AE[n] = np.load(ROOT / "tiles" / f"{area}_{n}.npz")["ae"].astype(np.float32)
rng = np.random.default_rng(SC["seed"])
rows, truth_store = [], {}
for pool, qarea in POOL_Q.items():
    tiles = TILES[pool]["test"]
    ae_pool = np.concatenate([AE[n].reshape(64, -1)[:, T[n]["valid"].ravel()].T for n in tiles])
    clim_pool = LT[np.concatenate([T[n]["cell"][T[n]["valid"]] for n in tiles])]
    for q in Q:
        ref = q["plan"]
        if ref["area"] != qarea:
            continue
        plan = dict(ref, area=pool)
        g = answer(plan, pool)
        truth_store[f"{q['id']}|{pool}"] = np.packbits(g)
        row = {
            "id": q["id"],
            "pool": pool,
            "heldout": pool not in SEEN,
            "persona": q["persona"],
            "n_extra": n_extra(ref),
            "has_climate": bool(ref["climate"]),
            "output": ref["output"],
            "truth_px": int(g.sum()),
            "n_px": len(g),
            "prevalence": float(g.mean()),
            "F1_land": f1(answer(plan, pool, "ae"), g),
            "F1_planted": f1(answer(plan, pool, "planted"), g),
        }
        # places like this point: 5 answer pixels from the training tiles of the question's own area
        tr = TILES[qarea]["train"]
        got = execute(ref, {n: T[n]["official"] for n in tr}, CTX[qarea])
        cand = [(n, i) for n in tr for i in np.flatnonzero((got[n] & T[n]["valid"]).ravel())]
        if len(cand) >= LP["examples"] and g.any():
            pick = [cand[i] for i in rng.choice(len(cand), LP["examples"], replace=False)]
            ex_ae = np.stack([AE[n].reshape(64, -1)[:, i] for n, i in pick])
            ex_cl = LT[[T[n]["cell"].ravel()[i] for n, i in pick]]
            s = like_point_scores(ae_pool, clim_pool, ex_ae, ex_cl, LP["climate_weight"])
            row["AP_like_point"] = average_precision_score(g, s)
        if ref["climate"]:
            clim = np.concatenate(
                [
                    np.logical_and.reduce(
                        [CTX[pool].climate(c)[T[n]["iy"], T[n]["ix"]] for c in ref["climate"]]
                    )[T[n]["valid"]]
                    for n in tiles
                ]
            )
            row["climate_true_share"] = float(clim.mean())
            row["power_cells"] = len(
                {(b, i) for n in tiles for b, i in zip(T[n]["iy"].ravel(), T[n]["ix"].ravel(), strict=True)}
            )
        rows.append(row)
    print(f"{pool}: done ({time.time() - T_START:.0f}s)")
land = pd.DataFrame(rows)
land["scored"] = land.truth_px >= SC["min_truth"]
land.to_csv(RES / "04_land.csv", index=False)
np.savez_compressed(ROOT / "maps" / "truth.npz", **truth_store)
print(
    f"{len(land)} question-pool pairs; scored {int(land.scored.sum())}; not scored:",
    land.loc[~land.scored, ["id", "pool", "truth_px"]].values.tolist(),
)

north_dakota: done (21s)


north_carolina: done (28s)


minnesota: done (39s)


georgia: done (45s)


116 question-pool pairs; scored 89; not scored: [['nd13', 'north_dakota', 0], ['nd14', 'north_dakota', 0], ['nc08', 'north_carolina', 0], ['nc09', 'north_carolina', 0], ['nc12', 'north_carolina', 0], ['nc13', 'north_carolina', 0], ['nc14', 'north_carolina', 0], ['nc17', 'north_carolina', 0], ['nc21', 'north_carolina', 0], ['nd06', 'minnesota', 0], ['nd07', 'minnesota', 0], ['nd08', 'minnesota', 0], ['nd10', 'minnesota', 0], ['nd11', 'minnesota', 0], ['nd12', 'minnesota', 0], ['nd13', 'minnesota', 0], ['nd14', 'minnesota', 0], ['nd17', 'minnesota', 0], ['nd22', 'minnesota', 0], ['nd33', 'minnesota', 0], ['nd35', 'minnesota', 0], ['nd37', 'minnesota', 0], ['nc09', 'georgia', 0], ['nc12', 'georgia', 0], ['nc13', 'georgia', 0], ['nc17', 'georgia', 0], ['nc21', 'georgia', 0]]


## D0, L1, L2

In [3]:
from askmap.score import median_ci

B, S = SC["bootstrap"], SC["seed"]
seen = land[land.scored & ~land.heldout]
held = land[land.scored & land.heldout]
d0 = median_ci(seen.F1_planted.to_numpy(), B, S)
l1 = median_ci(seen.F1_land.to_numpy(), B, S)
l2 = median_ci(held.F1_land.to_numpy(), B, S)
V = {
    "D0": {"pass": d0[0] >= 0.95, "median_F1": d0, "n": len(seen)},
    "L1": {"pass": l1[0] >= 0.60, "median_F1": l1, "n": len(seen)},
    "L2": {"pass": l2[0] >= 0.50, "median_F1": l2, "n": len(held)},
    "like_point_median_AP": {
        "seen": float(seen.AP_like_point.median()),
        "heldout": float(held.AP_like_point.median()),
        "median_chance": float(seen.prevalence.median()),
    },
    "minutes": round((time.time() - T_START) / 60, 1),
}
(RES / "04_land.json").write_text(json.dumps(V, indent=1, default=float))
for k in ("D0", "L1", "L2"):
    print(k, "PASS" if V[k]["pass"] else "FAIL", {kk: vv for kk, vv in V[k].items() if kk != "pass"})
print("places like this point (information):", V["like_point_median_AP"])

D0 FAIL {'median_F1': (0.7814504994721027, 0.6438682022657152, 0.9238617847433087), 'n': 49}
L1 PASS {'median_F1': (0.6847324216173432, 0.46685082872928174, 0.8067224794599567), 'n': 49}
L2 FAIL {'median_F1': (0.48891658506187124, 0.4209734854708887, 0.6111189572348645), 'n': 40}
places like this point (information): {'seen': 0.38942443348013606, 'heldout': 0.16769629672951625, 'median_chance': 0.05554419192202999}


In [4]:
qtxt = {q["id"]: q["question"] for q in Q}
pd.set_option("display.max_colwidth", 110)
show = land.assign(question=land.id.map(qtxt))
show[
    ["id", "pool", "scored", "truth_px", "prevalence", "F1_land", "F1_planted", "AP_like_point", "question"]
].sort_values(["pool", "F1_land"]).round(3)

,id,pool,scored,truth_px,prevalence,F1_land,F1_planted,AP_like_point,question
101,nc07,georgia,True,1546,0.001,0.066,0.365,0.028,"In the North Carolina study area, show solar farms within 100 m of forest."
98,nc04,georgia,True,232,0.000,0.111,0.000,0.000,"In the North Carolina study area, show patches of grassland and pasture of at least 10 ha within 2 km of a..."
108,nc14,georgia,True,241,0.000,0.143,0.076,NaN,"In the North Carolina study area, show soybean fields where September 2024 rainfall was in the wettest 10%..."
112,nc18,georgia,True,3725,0.003,0.286,0.066,0.005,"In the North Carolina study area, show wetlands within 500 m of an existing solar farm."
97,nc03,georgia,True,34822,0.026,0.347,0.288,0.114,"In the North Carolina study area, show grassland and pasture within 2 km of an existing solar farm."
...,...,...,...,...,...,...,...,...,...
6,nd07,north_dakota,True,431260,0.323,0.926,0.957,0.677,"In the North Dakota study area, show cropland where April-June 2022 rainfall was in the wettest 10% of spr..."
7,nd08,north_dakota,True,431260,0.323,0.926,0.957,0.687,"In the North Dakota study area, show cropland where root-zone soil wetness in April-May 2022 was in the we..."
24,nd25,north_dakota,True,431260,0.323,0.926,0.957,0.759,"In the North Dakota study area, show cropland where May-September 2024 rainfall was in the wettest 20% of ..."
12,nd13,north_dakota,False,0,0.000,1.000,1.000,NaN,How much grassland and pasture in the North Dakota study area had July-August 2021 rainfall in the driest ...


In [5]:
cl = land[land.has_climate]
flat = int(cl.climate_true_share.isin([0, 1]).sum())
print(
    f"Questions with climate conditions: POWER cells under a pool's 3 test tiles: "
    f"{cl.power_cells.min()}-{cl.power_cells.max()}; share of pool pixels meeting the climate part: "
    f"median {cl.climate_true_share.median():.2f}, all-or-nothing in {flat} "
    f"of {len(cl)} question-pool pairs"
)

Questions with climate conditions: POWER cells under a pool's 3 test tiles: 4.0-6.0; share of pool pixels meeting the climate part: median 0.49, all-or-nothing in 39 of 82 question-pool pairs
